<h1 align="center">
B Term research notes
</h1>

1. A term recap .............................................................................................................................................................................................





# 1. Recap

In investigatng the connection between dynamical systems and MLPs the main use case of interest so far has been denoising Auto encoders applied to MNST images with gaussian noise applied. The larger topics covered so far in the resesarch of this MQP as well as relevant research from the Paffenroth research group includes:
1. RNNs are just discrete time dynamical systems and can be represented by a blockwise matrix of functions(Hershy et. al).
2. Technically all MLPs are discretions of Euler steps applied to a vector of n dimensions where n is the max width of the network layers.(MQP generated)
3. 

# Diagnosing attractors

## Formal Definition of an attractor for a discrete time dynamical system

For discrete dymamical system. 

$$x_{k+1} = f(x_{k}) \qquad x_{k} \epsilon \mathbb{R}^{n}$$

an attractor $A \subset \mathbb{R}^n$ is a set satisfying
1. Invarience: $f(A) = A$ if you are at the point of the attractor you stay there.
2. Attraction: There exists a basin of attraction $B(A)$ such that: 
$$\forall x_0 \epsilon B(A), \lim{k \to \infty} dist(x_k, A) = 0$$ in otherwords trajectories close the the attractor converge to it.$$
3. Minimality: There's no smaller closed invariant set that attracts the same basin.

Attractors can be:
- Fixed points: $X^* = f(x^*)$
- Limit Cycles: periodic orbits, $f^{p}(x) = x$
- chaotic attractors: invariant, bounded, fractal-like sets with positive Lyapunov exponets

## detecting attractors

(A) Fixed Point / Cycle Tests

1. Fixed Points: Solve $F(x) - x = 0$
2. Limit Cycles: Detect repeating states $\| x_{k+p} - x_k\| < \epsilon$ for some tolerance $\epsilon$ and some period $p > 1$ 




In [2]:
import torch
from dataclasses import dataclass
from typing import Callable, Optional, Tuple

def detect_fixed_point(traj: torch.Tensor, eps: float = 1e-4) -> Optional[int]:
    """
    Detect first t with ||x_{t+1} - x_t||_2 < eps (approx fixed point).
    Returns the index t (0-based), or None if not found.
    """
    diffs = torch.norm(traj[1:] - traj[:-1], dim=1)
    idx = (diffs < eps).nonzero(as_tuple=False)
    return int(idx[0].item()) if idx.numel() > 0 else None


def detect_cycle(
    traj: torch.Tensor,
    max_period: int = 10,
    eps: float = 1e-3,
) -> Tuple[Optional[int], Optional[int]]:
    """
    Search for short limit cycles by checking ||x_t - x_{t-p}||_2 < eps for 2 <= p <= max_period.
    Returns (cycle_start_index, period) or (None, None) if not found.
    """
    T = traj.size(0)
    for p in range(2, max_period + 1):
        for t in range(p, T):
            if torch.norm(traj[t] - traj[t - p]) < eps:
                # Optionally backtrack one period to mark earliest cycle index
                return t - p, p
    return None, None


(B) Local Stability (Jacobian/Linearization)

at any fixed point $x^*$: 

1. Compute the Jacobian $J = Df(x^*)$
2. Find its eigenvalues $\lambda_{i}$

- Stable(attracting): $|\lambda_{i} < 1$ for all $i$
- Unstable (repelling): any $|\lambda_{i}| > 1$
- Marginal: some $|\lambda_{i}| = 1$

(D) Lyapunov exponent estimation
If trajectories don't converge to fixed points or cycles, compute the largest Lyapunov exponent:
$$

\lambda_{\max} = \lim_{k \to \infty} \frac{1}{k} \sum_{i=0}^{k-1} \ln \frac{\|\delta x_{i+1}\|}{\|\delta x_i\|}
$$
Where $\delta x_{i+1} \approx J_{f}(x_i)\delta x_i$


# Week of the 29th Tests

1) Train a normal denoising auto encoder and test it on many iterations. Record multiple trajectories.
    Each trajectory is produced by applying the same trained DNAE to a different starting image and plotting the state over "time".
    
    Apply attractor, chaotic, and biffurication tests.

2) Try different training configurations (i.e. train it on multiple refeeds, change the loss to enforce improvement each iteration)
    see if attractor behavior changes

3) optional: try to compare the training of a single fixed dimensional vector trained on x refeeds

In [ ]:
def jvp(f: Callable[[torch.Tensor], torch.Tensor], x: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """
    Compute J_f(x) @ v (same shape as x) using autograd.
    x, v have shape (1, D). Returns (1, D).
    """
    x = x.detach().requires_grad_(True)
    y = f(x)
    assert y.shape == x.shape, "f must be R^D -> R^D (pointwise denoiser)."
    dot = (y * v).sum()
    (grad_x,) = torch.autograd.grad(dot, x, retain_graph=True, create_graph=False)
    return grad_x


def vjp(f: Callable[[torch.Tensor], torch.Tensor], x: torch.Tensor, u: torch.Tensor) -> torch.Tensor:
    """
    Compute J_f(x)^T @ u (same shape as x) using autograd.
    x, u have shape (1, D). Returns (1, D).
    """
    x = x.detach().requires_grad_(True)
    y = f(x)
    (grad_x,) = torch.autograd.grad(outputs=y, inputs=x, grad_outputs=u, retain_graph=True, create_graph=False)
    return grad_x


def spectral_norm_of_jacobian(
    f: Callable[[torch.Tensor], torch.Tensor],
    x_star: torch.Tensor,
    iters: int = 50,
    device: str = "cpu",
) -> float:
    """
    Estimate ||J_f(x*)||_2 (largest singular value) via power iteration on J^T J, without
    materializing J. If < 1, f is locally a contraction at x* (sufficient condition).
    """
    with torch.no_grad():
        x = x_star.view(1, -1).to(device)

    u = torch.randn_like(x)
    u = u / (u.norm() + 1e-12)

    for _ in range(iters):
        # v = (J^T J) u  ≈  J^T (J u)
        w = jvp(f, x, u)       # w = J u
        v = vjp(f, x, w)       # v = J^T w
        v_norm = v.norm() + 1e-12
        u = v / v_norm

    # Rayleigh quotient: ||J u||_2 is the singular value estimate
    Ju = jvp(f, x, u)
    return float(Ju.norm().item())
